# NB105 - Sprint-1 lane-head ablation (data-corrected by NB104)

Small, composable changes to the NB101 polyline head, ablated on the 10k subset
(~15-ep probes). **S1.1 confidence threshold (tau=0.4) is BANKED on every row**
(NB104 S0.3: free +13% F1 + fixes "always 8 lanes"); the rows test the training
levers that still have a rationale after Sprint 0.

**Round 2 (lean re-run).** Round 1 hit a bug: S1.3's QFL used the *raw* matched
line-IoU (~0.05 early) as the target, collapsing score_std 0.082->0.005 -> tau
rejected all -> F1=0. FIXED (soft target = floor 0.5 + quality; regression-
tested). Round 1 also showed y-reweight 'near' HURT F1 (abandoned the scoring
far-field). So round 2 drops the broken combos and the aggressive 'near':

| row | S1.3 IoU-cls (fixed) | S1.4 y-reweight | tests |
|---|---|---|---|
| `s1_baseline`      | off | none  | **REUSED from round 1** (F1=0.148) - not re-run (loss byte-identical with levers off) |
| `s1b_iou_cls`      | 2.0 | none  | does the FIXED QFL spread score_std + lift F1? |
| `s1b_iou_mildyrw`  | 2.0 | angle | + mild ('angle' 0.45/0.55) reweight, not the hurtful 'near' |

> Only 2 GPU rows this round - the baseline is reused, so you launch
> `s1b_iou_cls` and `s1b_iou_mildyrw` only.

**Decision gate G1:** if a row reaches **F1 >= ~0.32, score_std past 0.09,
length_mean stops shrinking, mAP50 >= 0.48** -> promote it to 70k. If F1 still
ceilings < 0.30 despite threshold-able cls -> representation is the wall ->
Sprint 2 (CLRerNet LaneIoU).

> Levers are env-driven in MTDETRDLoss (`LANE_IOU_CLS`, `LANE_Y_REWEIGHT`,
> `LANE_SMOOTH_W`, `LANE_EVAL_TAU`), set by train_lane_only's new CLI flags. All
> default OFF => byte-identical to NB101 when unset. Helper math is unit-tested
> (tests/test_sprint1_levers.py).


### Cell 1: Mount + deps + locate (mount-alive guarded)

In [1]:
import os, sys, subprocess
from pathlib import Path
os.environ['PYTHONIOENCODING'] = 'utf-8'

REPO_ROOT  = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
MIG        = Path(REPO_ROOT)/'stage2/rmt_ppad_migration'
TRAIN      = MIG/'P8_train/scripts/train_lane_only.py'
MODEL      = MIG/'vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml'
DATA       = MIG/'vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only_10k.yaml'
DATASETS   = Path('/content/drive/MyDrive/EcoCAR/datasets')
SUBSET     = Path('/content/bdd_subset_10k')

def _alive():
    try: return os.path.isdir('/content/drive/MyDrive')
    except OSError: return False
if not _alive():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
    except Exception as e: print('[mount]', e)
if not _alive():
    raise RuntimeError('Drive mount DEAD (OSError 107). Runtime -> Disconnect '
                       'and delete runtime, reconnect, re-run from Cell 1.')
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
for _p in ('addict','yapf','scipy'):
    try: __import__(_p)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',_p])
try: import mmcv
except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q','mmcv'])
for p in (TRAIN, MODEL, DATA):
    assert p.exists(), f'missing {p} (sync Drive)'
print('[ok] env ready')


Mounted at /content/drive
[ok] env ready


### Cell 2: Extract the 10k subset + point the data YAML at it
Idempotent. Same subset NB98/NB92 used (lane-only; drivable removed for the
clean collate path).


In [2]:
PREP = MIG/'extensions/bezier_lcm/scripts/prepare_bdd_subset_10k.py'
req = [DATASETS/'bdd100k_clrkd_curve.tar',
       Path('/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip'),
       DATASETS/'lane_targets_clr_v1_polyline.tar.gz']
missing=[str(p) for p in req if not p.exists()]
if missing: raise FileNotFoundError('Missing:\n  '+'\n  '.join(missing))
if (SUBSET/'prep_summary.json').exists() or (
        (SUBSET/'images/val2017').exists() and any((SUBSET/'images/val2017').iterdir())):
    print('[ok] subset present; skipping')
else:
    cmd=[sys.executable,'-u',str(PREP),'--out-root',str(SUBSET),
         '--n-train','10000','--n-val','2000','--seed','89']
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait()!=0: raise RuntimeError('subset prep failed')

def _yset(y, f, v):
    txt=y.read_text(encoding='utf-8'); line=f'{f}: {v}'
    if line in txt: return
    L=txt.splitlines()
    for i,ln in enumerate(L):
        if ln.strip().startswith(f'{f}:'): L[i]=line; break
    else: L.append(line)
    y.write_text('\n'.join(L)+'\n',encoding='utf-8'); print(f'  [yaml] {f} -> {v}')
def _yunset(y, f):
    txt=y.read_text(encoding='utf-8')
    keep=[ln for ln in txt.splitlines() if not ln.strip().startswith(f'{f}:')]
    n='\n'.join(keep)+'\n'
    if n!=txt: y.write_text(n,encoding='utf-8'); print(f'  [yaml] removed {f}')
_yset(DATA,'path','/content/bdd_subset_10k')
_yset(DATA,'lane_targets_root','/content/bdd_subset_10k/lane_targets')
_yunset(DATA,'drivable_masks_root')
print('[ok] data yaml configured')


[NB89.prep10k] out_root=/content/bdd_subset_10k n_train=10000 n_val=2000
[prep] step 1: extract curve tarball
  extracting bdd100k_clrkd_curve.tar -> /content/bdd_curve_scratch
  done in 101.7s
  curve images: train=/content/bdd_curve_scratch/images/train (70000 jpgs), val=/content/bdd_curve_scratch/images/val (10000 jpgs)
[prep] step 2: hardlink image subset
  sampled 10000 train + 2000 val stems
  on-disk: 10000 train, 2000 val
[prep] step 3: extract matching detection labels
  labels zip: 80003 entries; first 3: ['labels/', 'labels/val2017/', 'labels/val2017/c8b4e0ea-9581068d.txt']
  extracted labels: train=10000 val=2000
[prep] step 4: extract matching lane_targets .pt files
  lane tar first entries: ['lane_targets/train2017/adfc8f5c-8bd2f72d.pt', 'lane_targets/train2017/2beccce2-18444154.pt', 'lane_targets/train2017/97f0a30d-9e0685bf.pt']
  extracted lane targets: train=10000 val=2000

[prep] final inventory:
  images/train2017               10000
  images/val2017                 

### Cell 3: `launch(name, **levers)` helper

Every row uses the NB101 recipe (clrkd, hungarian, clamp=100, fliplr=0.5,
wd=0.05) + **tau=0.4 eval threshold banked**, varying only the S1 training
levers. Resume-safe; best.pt/last.pt/full_train.log sync to Drive each epoch.


In [3]:
from stage2.scripts.notebook_utils import run_streaming
PROJECT = '/content/runs/sprint1'
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)
EPOCHS, BATCH, LR0, PATIENCE = 15, 32, '4e-4', 99   # 15-ep probe, no early-stop

def launch(name, iou_cls='off', y_reweight='none', smooth_w=0.0, tau=0.4):
    cmd=[sys.executable,'-u',str(TRAIN),
         '--mode','full','--model-yaml',str(MODEL),'--data-yaml',str(DATA),
         '--project',PROJECT,'--name',name,'--device','0',
         '--save-period','15','--batch',str(BATCH),'--epochs',str(EPOCHS),
         '--lr0',LR0,'--patience',str(PATIENCE),
         '--fliplr','0.5','--weight-decay','0.05',
         '--lane-match','hungarian','--lane-weights','clrkd','--diff-clamp','100',
         # S1 levers under test:
         '--lane-iou-cls',str(iou_cls),'--lane-y-reweight',str(y_reweight),
         '--lane-smooth-w',str(smooth_w),'--lane-eval-tau',str(tau)]
    print(f'\n=== {name}: iou_cls={iou_cls} y_reweight={y_reweight} '
          f'smooth_w={smooth_w} tau={tau} ===\n', flush=True)
    rc=run_streaming(cmd, log_path=os.path.join(LOG_DIR,f'NB105_{name}.log'), check=False)
    print(f'[{name}] rc={rc}  (watch lane_f1, lane_score_std, length_mean, mAP50)')
    return rc==0
print('[ready] call launch(...) per row cell below')


[ready] call launch(...) per row cell below


### Cell 4: Row 0: baseline - REUSE round-1 weights (do NOT re-run)

In [4]:
# Baseline is REUSED from round 1, not re-launched. All my loss edits are
# gated behind iou_cls/y_reweight/smooth flags; with them OFF the loss is
# byte-identical to round 1, so s1_baseline's result (F1=0.148, score_std
# =0.082, mAP50=0.671 @15ep) is unchanged. The aggregator reads the
# existing /content/runs/sprint1/s1_baseline/results.csv (or the recorded
# numbers as a fallback). Nothing to run here.
print('[baseline] reusing round-1 s1_baseline (F1=0.148) - not re-run')


[baseline] reusing round-1 s1_baseline (F1=0.148) - not re-run


### Cell 5: Row 1: FIXED S1.3 IoU-aware cls alone (QFL gamma=2.0, floor=0.5)

In [5]:
launch('s1b_iou_cls', iou_cls='2.0')


流式输出内容被截断，只能显示最后 5000 行内容。
       1/15        50G      40.36          0      25.57  6.415e-05      113.4      0.504     0.5019        388        640:  75%|███████▍  | 234/313 [02:30<00:46,  1.71it/s]
       1/15        50G      40.31          0      25.52  6.441e-05      114.2     0.5044      0.502        359        640:  75%|███████▍  | 234/313 [02:31<00:46,  1.71it/s]
       1/15        50G      40.27          0      25.48  6.466e-05      116.2     0.5043     0.5019        319        640:  75%|███████▍  | 234/313 [02:32<00:46,  1.71it/s]
       1/15        50G      40.22          0      25.44  6.492e-05      128.7     0.5044     0.5019        371        640:  75%|███████▍  | 234/313 [02:32<00:46,  1.71it/s]
       1/15        50G      40.17          0      25.41  6.518e-05        129     0.5043     0.5021        333        640:  75%|███████▍  | 234/313 [02:33<00:46,  1.71it/s]
       1/15        50G      40.13          0      25.37  6.543e-05        124     0.5042     0.5019        2

True

### Cell 6: Row 2: FIXED iou_cls + MILD y-reweight ('angle', not 'near')

In [6]:
launch('s1b_iou_mildyrw', iou_cls='2.0', y_reweight='angle')


流式输出内容被截断，只能显示最后 5000 行内容。
       1/15        50G      40.16          0      25.61   6.39e-05      103.3     0.5044     0.5024        318        640:  75%|███████▌  | 236/313 [02:29<00:45,  1.69it/s]
       1/15        50G      40.11          0      25.57  6.415e-05      113.8     0.5042     0.5023        388        640:  75%|███████▌  | 236/313 [02:30<00:45,  1.69it/s]
       1/15        50G      40.06          0      25.53  6.441e-05      108.7     0.5048     0.5024        359        640:  75%|███████▌  | 236/313 [02:30<00:45,  1.69it/s]
       1/15        50G      40.01          0      25.48  6.466e-05      111.2     0.5045     0.5023        319        640:  75%|███████▌  | 236/313 [02:31<00:45,  1.69it/s]
       1/15        50G      39.97          0      25.45  6.492e-05      126.6     0.5048     0.5023        371        640:  75%|███████▌  | 236/313 [02:31<00:45,  1.69it/s]
       1/15        50G      39.92          0      25.42  6.518e-05      125.5     0.5046     0.5023        3

True

### Cell 9: Aggregate - rank rows by lane_f1 (@tau=0.4) + check the guardrail

Reads each row's results.csv final epoch. WINNER = highest lane_f1 with
mAP50 >= 0.48 AND score_std visibly > 0.09 (cls now threshold-able) AND
length_mean not shrinking. That row is promoted to 70k.


In [7]:
import csv
from pathlib import Path
PROJECT = Path('/content/runs/sprint1')
DRIVE_CK = Path('/content/drive/MyDrive/EcoCAR/training_runs/checkpoints')
ROWS = ['s1_baseline','s1b_iou_cls','s1b_iou_mildyrw']
# Baseline reused from round 1: its result is unchanged (loss byte-identical
# with levers off). Hard-coded fallback = the round-1 final-epoch numbers so the
# table is complete even if /content was wiped and the folder isn't re-synced.
_BASELINE_R1 = {'metrics/lane_f1(lane)': '0.1483',
                'metrics/lane_curveIoU(lane)': '0.5166',
                'metrics/lane_score_std(lane)': '0.0824',
                'metrics/lane_length_mean(lane)': '0.1052',
                'metrics/mAP50(B)': '0.6712'}
def final(name):
    # search /content run dir, then the Drive checkpoints mirror.
    for base in (PROJECT/name, DRIVE_CK/name):
        p=base/'results.csv'
        if p.exists():
            rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(open(p))]
            if rows: return rows[-1]
    if name=='s1_baseline':
        return _BASELINE_R1     # reuse round-1 numbers (loss unchanged)
    return None
def g(r,*ks):
    for k in ks:
        for ck in (r or {}):
            if ck.strip().lower()==k.lower():
                try: return float(r[ck])
                except: pass
    return None
print(f'{"row":15} {"F1":>7} {"curveIoU":>9} {"score_std":>10} {"len_mean":>9} {"mAP50":>7} {"verdict":>9}')
print('-'*72)
best=None
for n in ROWS:
    r=final(n)
    if r is None: print(f'{n:15} (not run)'); continue
    f1=g(r,'metrics/lane_f1(lane)'); ci=g(r,'metrics/lane_curveIoU(lane)')
    ss=g(r,'metrics/lane_score_std(lane)'); lm=g(r,'metrics/lane_length_mean(lane)')
    mp=g(r,'metrics/mAP50(B)')
    ok = (f1 or 0)>=0.32 and (mp or 0)>=0.48
    def fmt(x): return f'{x:.4f}' if isinstance(x,float) else '  -  '
    print(f'{n:15} {fmt(f1):>7} {fmt(ci):>9} {fmt(ss):>10} {fmt(lm):>9} {fmt(mp):>7} {("G1 PASS" if ok else "-"):>9}')
    if f1 is not None and (mp or 0)>=0.48 and (best is None or f1>best[1]): best=(n,f1)
print()
if best: print(f'WINNER (mAP50>=0.48): {best[0]}  lane_f1={best[1]:.4f}')
print('G1: F1>=0.32 + score_std>0.09 + length not shrinking + mAP50>=0.48 -> promote to 70k.')
print('If F1 ceilings <0.30 with threshold-able cls -> Sprint 2 (CLRerNet LaneIoU).')


row                  F1  curveIoU  score_std  len_mean   mAP50   verdict
------------------------------------------------------------------------
s1_baseline      0.1483    0.5166     0.0824    0.1052  0.6712         -
s1b_iou_cls      0.0000    0.5296     0.0224    0.0784  0.6976         -
s1b_iou_mildyrw  0.0000    0.5343     0.0244    0.0837  0.6940         -

WINNER (mAP50>=0.48): s1_baseline  lane_f1=0.1483
G1: F1>=0.32 + score_std>0.09 + length not shrinking + mAP50>=0.48 -> promote to 70k.
If F1 ceilings <0.30 with threshold-able cls -> Sprint 2 (CLRerNet LaneIoU).
